# logistic regression
Regress amplification on somatic mutation burden, tumor type, batch.  

In [ ]:
#import cyvcf2
import pandas as pd
import numpy as np
import os
from pathlib import Path
#import sklearn
import matplotlib.pyplot as plt

import sys
sys.path.append('../src')
from data_imports import *
from somatic_snv_functions import *

OUT_DIR=Path('out')
OUT_DIR.mkdir(parents=True,exist_ok=True)

In [ ]:
df = merge_ecDNA_counts(import_biosamples(),
                   reindex_counts(read_manifest(),get_variant_count_df())
                  )
df.head()
# write df, including duplicates, then drop duplicates
path=OUT_DIR/'biosample_mutation_burden.tsv'
df.to_csv(path,sep='\t')

df = df[df.in_snv_set]
df.head()

## Test
logistic regression  
forest plot

In [ ]:
import matplotlib.pyplot as plt
#from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

In [ ]:
def logistic_regression_forest_plot(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"] # RBL RMS/RHB OST ETMR
    df = df[df["cancer_type"].isin(target_types)]
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 0 if x == 'no amplification' else 1)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    # TODO standardize counts (use z-score)
    # (optional) sex, age (z-score)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    X = sm.add_constant(X) 
    y = df_encoded['amplicon_binary']
    X = X.astype(float)
    y = y.astype(float)
    model = sm.Logit(y, X).fit(disp=False)
    print(model.summary())
    
    params = model.params
    conf = model.conf_int()
    conf["OR"] = params
    conf.columns = ["2.5%", "97.5%", "OR"]
    conf = np.exp(conf) 
    conf = conf.loc[[c for c in conf.index if c != "const"]]
    conf = conf.sort_values("OR", ascending=True)
    
    # ---- Forest plot ----
    # TODO add batch, tumor burden to forest plot
    plt.figure(figsize=(6, 4))
    plt.errorbar(conf["OR"], conf.index, 
                 xerr=[conf["OR"] - conf["2.5%"], conf["97.5%"] - conf["OR"]],
                 fmt="o", color="black", ecolor="gray", elinewidth=2, capsize=4)
    plt.axvline(x=1, color="red", linestyle="--")
    plt.xlabel("Odds Ratio (log scale)")
    plt.xscale("log")
    plt.title("Association between Tumor burden and Amplicon Presence")
    plt.tight_layout()
    plt.show()
    
    return model

In [ ]:
logistic_regression_forest_plot(df_all)

In [ ]:
def logistic_regression_ecDNA_vs_intra_forest_plot(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"]
    df = df[df["cancer_type"].isin(target_types)]
    df = df[df['amplicon_class'] != 'no_amplification'] 
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 1 if x == 'ecDNA' else 0)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    X = sm.add_constant(X) 
    y = df_encoded['amplicon_binary']
    X = X.astype(float)
    y = y.astype(float)
    model = sm.Logit(y, X).fit(disp=False)
    print(model.summary())
    
    params = model.params
    conf = model.conf_int()
    conf["OR"] = params
    conf.columns = ["2.5%", "97.5%", "OR"]
    conf = np.exp(conf) 
    conf = conf.loc[[c for c in conf.index if c != "const"]]
    conf = conf.sort_values("OR", ascending=True)
    
    # ---- Forest plot ----
    plt.figure(figsize=(6, 4))
    plt.errorbar(conf["OR"], conf.index, 
                 xerr=[conf["OR"] - conf["2.5%"], conf["97.5%"] - conf["OR"]],
                 fmt="o", color="black", ecolor="gray", elinewidth=2, capsize=4)
    plt.axvline(x=1, color="red", linestyle="--")
    plt.xlabel("Odds Ratio (log scale)")
    plt.xscale("log")
    plt.title("Association between Tumor burden and Amplicon Presence")
    plt.tight_layout()
    plt.show()
    
    return model

In [ ]:
logistic_regression_ecDNA_vs_intra_forest_plot(df_all)

## practice

In [ ]:
def sm_logistic_regression(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"]
    df = df[df["cancer_type"].isin(target_types)]
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 0 if x == 'no amplification' else 1)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    X = sm.add_constant(X) 
    y = df_encoded['amplicon_binary']
    X = X.astype(float)
    y = y.astype(float)
    model = sm.Logit(y, X).fit(disp=False)
    print(model.summary())

    return model, X, y

In [ ]:
def sm_logistic_regression_ecDNA_vs_intra(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"]
    df = df[df["cancer_type"].isin(target_types)]
    df = df[df['amplicon_class'] != 'no_amplification'] 
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 1 if x == 'ecDNA' else 0)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    X = sm.add_constant(X) 
    y = df_encoded['amplicon_binary']
    X = X.astype(float)
    y = y.astype(float)
    model = sm.Logit(y, X).fit(disp=False)
    print(model.summary())

    return model, X, y

In [ ]:
def sklearn_logistic_regression(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"]
    df = df[df["cancer_type"].isin(target_types)]
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 0 if x == 'no amplification' else 1)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    y = df_encoded['amplicon_binary']
    model = LogisticRegression()
    model.fit(X, y)
    #print("Coefficients:", model.coef_)
    #print("Intercept:", model.intercept_)
    #print("Feature names:", X.columns.tolist())
    coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
    }).sort_values("Coefficient", ascending=False)
    print(coef_df)
    return model, X, y

In [ ]:
sklearn_logistic_regression(df_all)

In [ ]:
def sklearn_logistic_regression_ecDNA_vs_intra(df):
    target_types = ["CPT", "EPN", "GCT", "GNT", "HGG", "NBL", "MBL", "PINT", "SARC"]
    df = df[df["cancer_type"].isin(target_types)]
    df = df[df['amplicon_class'] != 'no_amplification'] 
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 1 if x == 'ecDNA' else 0)
    df_encoded = pd.get_dummies(df, columns=['cancer_type', 'batch'], drop_first=True)
    X = df_encoded[['all_counts'] + [col for col in df_encoded.columns if col.startswith('cancer_type_') or col.startswith('batch_')]]
    y = df_encoded['amplicon_binary']
    model = LogisticRegression()
    model.fit(X, y)
    #print("Coefficients:", model.coef_)
    #print("Intercept:", model.intercept_)
    #print("Feature names:", X.columns.tolist())
    coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
    }).sort_values("Coefficient", ascending=False)
    print(coef_df)
    return model, X, y

In [ ]:
m,x,y = sklearn_logistic_regression_ecDNA_vs_intra(df_all)

In [ ]:
m.coef_

In [ ]:
def plot_logistic_by_tumor_type(df):
    """
    Plot logistic regression curves for each tumor type.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing columns:
        - 'amplicon_class' (0 for no amplification, 1 for ecDNA/intrachromosomal)
        - 'all_counts'
        - 'cancer_type'
    """
    df["amplicon_binary"] = df["amplicon_class"].replace({
        "no amplification": 0,
        "ecDNA": 1,
        "intrachromosomal": 1
    })
    
    scaler = StandardScaler()
    df["somatic_variant_scaled"] = scaler.fit_transform(df[["all_counts"]])

    cancer_types = df["cancer_type"].unique()

    plt.figure(figsize=(8, 6))

    for ttype in cancer_types:
        subset = df[df["cancer_type"] == ttype]
        X = subset[["somatic_variant_scaled"]]
        y = subset["amplicon_binary"]

       
        if len(subset[y == 1]) > 0 and len(subset[y == 0]) > 0:
            model = LogisticRegression()
            model.fit(X, y)

            x_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
            y_pred = model.predict_proba(x_range)[:, 1]

            plt.plot(x_range, y_pred, label=ttype)

    plt.xlabel("Somatic variant counts (scaled)")
    plt.ylabel("Probability of amplification")
    plt.title("Logistic regression by cancer type")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_logistic_by_tumor_type(df)

In [ ]:
def logistic_regression_all_types(df):
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 0 if x == 'no amplification' else 1)

    cancer_types = df['cancer_type'].unique()

    plt.figure(figsize=(10, 7))

    for ctype in cancer_types:
        sub = df[df['cancer_type'] == ctype]
        # クラスが1種類しかない場合はスキップ
        if len(sub['amplicon_binary'].unique()) < 2:
            continue

        X = sub[['all_counts']]
        y = sub['amplicon_binary']

        model = LogisticRegression()
        model.fit(X, y)

        # 曲線を描画するためのX範囲を作成
        x_vals = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
        y_pred = model.predict_proba(x_vals)[:, 1]

        # データ点（薄く表示）
        plt.scatter(X, y, alpha=0.25, s=15)

        # 各cancer typeごとのロジスティック曲線
        plt.plot(x_vals, y_pred, linewidth=2, label=ctype)

    plt.xlabel('Somatic Variant Counts')
    plt.ylabel('Binary (0=no amplification, 1=ecDNA/intrachromosomal)')
    plt.title('Logistic Regression Curves by Cancer Type')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
logistic_regression_all_types(df)

In [ ]:
import matplotlib.pyplot as plt

def logistic_regression_all_types_ecDNA_vs_intra(df):
    # ecDNAとintrachromosomalのみ対象
    df = df[df['amplicon_class'].isin(['ecDNA', 'intrachromosomal'])].copy()
    df['amplicon_binary'] = df['amplicon_class'].apply(lambda x: 1 if x == 'ecDNA' else 0)

    cancer_types = df['cancer_type'].unique()

    plt.figure(figsize=(10, 7))

    for ctype in cancer_types:
        sub = df[df['cancer_type'] == ctype]
        # クラスが1種類しかない場合はスキップ
        if len(sub['amplicon_binary'].unique()) < 2:
            continue

        X = sub[['all_counts']]
        y = sub['amplicon_binary']

        model = LogisticRegression()
        model.fit(X, y)

        # 曲線を描画するためのX範囲を作成
        x_vals = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
        y_pred = model.predict_proba(x_vals)[:, 1]

        # データ点（薄く表示）
        plt.scatter(X, y, alpha=0.25, s=15)

        # 各cancer typeごとのロジスティック曲線
        plt.plot(x_vals, y_pred, linewidth=2, label=ctype)

    plt.xlabel('Somatic Variant Counts')
    plt.ylabel('Binary (0=intrachr, 1=ecDNA)')
    plt.title('Logistic Regression Curves by Cancer Type')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
logistic_regression_all_types_ecDNA_vs_intra(df)